**Import Library**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.preprocessing import StandardScaler, RobustScaler

# Statistical
from scipy import stats

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
np.random.seed(42)

print("All libraries imported successfully!")

All libraries imported successfully!


**Mounted Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Load Dataset**

In [3]:
df = pd.read_excel("/content/drive/MyDrive/Internship /data (1).xlsx")
df.head(5)

,time,Cyclone_Inlet_Gas_Temp,Cyclone_Material_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Gas_Outlet_Temp,Cyclone_Inlet_Draft
0,2017-01-01 00:00:00,867.63,910.42,-189.54,-186.04,852.13,-145.9
1,2017-01-01 00:05:00,879.23,918.14,-184.33,-182.1,862.53,-149.76
2,2017-01-01 00:10:00,875.67,924.18,-181.26,-166.47,866.06,-145.01
3,2017-01-01 00:15:00,875.28,923.15,-179.15,-174.83,865.85,-142.82
4,2017-01-01 00:20:00,891.66,934.26,-178.32,-173.72,876.06,-143.39


In [4]:
original_shape = df.shape
print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Time range: {df['time'].min()} to {df['time'].max()}")

   Shape: 377,719 rows × 7 columns
   Time range: 2017-01-01 00:00:00 to 2020-08-07 12:15:00


**Data Preprocessing**

In [5]:
print(f"\n Columns detected:")
for i, col in enumerate(df.columns, 1):
  print(f"   {i}. {col} ({df[col].dtype})")


 Columns detected:
   1. time (datetime64[ns])
   2. Cyclone_Inlet_Gas_Temp (object)
   3. Cyclone_Material_Temp (object)
   4. Cyclone_Outlet_Gas_draft (object)
   5. Cyclone_cone_draft (object)
   6. Cyclone_Gas_Outlet_Temp (object)
   7. Cyclone_Inlet_Draft (object)


In [6]:
# Identify feature columns (all except 'time')
feature_columns = [col for col in df.columns if col != 'time']
print(f"\n Feature columns identified: {len(feature_columns)}")

        # Convert columns to numeric
print("\n Converting columns to numeric...")
for col in feature_columns:
  df[col] = pd.to_numeric(df[col], errors='coerce')
print("   Conversion complete")

        # Check for missing values
missing_counts = df[feature_columns].isnull().sum()
total_missing = missing_counts.sum()


if total_missing > 0:
  print(f"\n Missing values detected: {total_missing:,} total")
  print("\n   Missing values per column:")
  for col in feature_columns:
    if missing_counts[col] > 0:
      pct = (missing_counts[col] / len(df)) * 100
      print(f"      • {col}: {missing_counts[col]:,} ({pct:.2f}%)")


 Feature columns identified: 6

 Converting columns to numeric...
   Conversion complete

 Missing values detected: 8,195 total

   Missing values per column:
      • Cyclone_Inlet_Gas_Temp: 1,320 (0.35%)
      • Cyclone_Material_Temp: 1,591 (0.42%)
      • Cyclone_Outlet_Gas_draft: 1,321 (0.35%)
      • Cyclone_cone_draft: 1,320 (0.35%)
      • Cyclone_Gas_Outlet_Temp: 1,321 (0.35%)
      • Cyclone_Inlet_Draft: 1,322 (0.35%)


**Handling Missing Values**

In [7]:
# Handle missing values using forward fill then backward fill
print("\n   Handling missing values (forward fill → backward fill)...")
df[feature_columns] = df[feature_columns].fillna(method='ffill').fillna(method='bfill')

# Check if any remain
remaining = df[feature_columns].isnull().sum().sum()
if remaining > 0:
    print(f"     {remaining} values still missing - dropping rows...")
    df = df.dropna(subset=feature_columns)
else:
    print("    All missing values handled")

# Remove duplicates
print("\n Checking for duplicate timestamps...")
duplicates = df.duplicated(subset=['time'], keep='first').sum()
if duplicates > 0:
    print(f"   Found {duplicates:,} duplicate timestamps - removing...")
    df = df.drop_duplicates(subset=['time'], keep='first')
else:
    print("   No duplicates found")

# Sort by time
df = df.sort_values('time').reset_index(drop=True)

# Basic statistics
print("\n Data Statistics:")
print(f"   Final shape: {df.shape[0]:,} rows    {df.shape[1]} columns")
print(f"   Date range: {df['time'].min()} to {df['time'].max()}")
print(f"   Duration: {(df['time'].max() - df['time'].min()).days} days")


   Handling missing values (forward fill → backward fill)...
    All missing values handled

 Checking for duplicate timestamps...
   No duplicates found

 Data Statistics:
   Final shape: 377,719 rows    7 columns
   Date range: 2017-01-01 00:00:00 to 2020-08-07 12:15:00
   Duration: 1314 days


**Input Pipeline Construction**

In [8]:
df = df.copy()
feature_columns = feature_columns
sequence_length = 12
test_size = 0.2
scaler = MinMaxScaler()
model = None
X_scaled = None
X_sequences = None
threshold = None

# Define parameters explicitly
epochs = 50
batch_size = 32
percentile = 95


        # Extract and scale features
X = df[feature_columns].values
X_scaled = scaler.fit_transform(X)

print(f"   Original shape: {X.shape}")
print(f"   Scaled range: [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")

        # Create sequences
sequences = []
for i in range(len(X_scaled) - sequence_length + 1):
  sequences.append(X_scaled[i:i + sequence_length])

X_sequences = np.array(sequences)
print(f"   Sequences created: {X_sequences.shape}")
print(f"   Format: ({len(sequences)} sequences, {sequence_length} steps, {len(feature_columns)} features)")

        # Split into train/validation
split_idx = int(len(X_sequences) * (1 - test_size))
X_train = X_sequences[:split_idx]
X_val = X_sequences[split_idx:]

print(f"\n   Training set: {X_train.shape[0]} sequences")
print(f"   Validation set: {X_val.shape[0]} sequences")

   Original shape: (377719, 6)
   Scaled range: [0.0000, 1.0000]
   Sequences created: (377708, 12, 6)
   Format: (377708 sequences, 12 steps, 6 features)

   Training set: 302166 sequences
   Validation set: 75542 sequences


**Training LSTM Auto-Encoder**

In [9]:
n_features = len(feature_columns)

        # Encoder
inputs = layers.Input(shape=(sequence_length, n_features))
encoded = layers.LSTM(128, activation='relu', return_sequences=True)(inputs)
encoded = layers.Dropout(0.2)(encoded)
encoded = layers.LSTM(64, activation='relu', return_sequences=False)(encoded)
encoded = layers.Dropout(0.2)(encoded)
encoded = layers.Dense(32, activation='relu')(encoded)

        # Decoder
decoded = layers.RepeatVector(sequence_length)(encoded)
decoded = layers.LSTM(64, activation='relu', return_sequences=True)(decoded)
decoded = layers.Dropout(0.2)(decoded)
decoded = layers.LSTM(128, activation='relu', return_sequences=True)(decoded)
decoded = layers.Dropout(0.2)(decoded)
decoded = layers.TimeDistributed(layers.Dense(n_features))(decoded)

        # Autoencoder
model = Model(inputs, decoded)
model.compile(optimizer='adam', loss='mse')

        # Print architecture
print("\n   Architecture Summary:")
print(f"      Input shape: ({sequence_length}, {n_features})")
print(f"      Encoder: LSTM(128) → Dropout → LSTM(64) → Dropout → Dense(32)")
print(f"      Bottleneck: 32 dimensions")
print(f"      Decoder: RepeatVector → LSTM(64) → Dropout → LSTM(128) → Dropout → Dense({n_features})")
print(f"      Total parameters: {model.count_params():,}")

print("-" * 80)
print("TRAINING LSTM AUTOENCODER")
print("-" * 80)


   Architecture Summary:
      Input shape: (12, 6)
      Encoder: LSTM(128) → Dropout → LSTM(64) → Dropout → Dense(32)
      Bottleneck: 32 dimensions
      Decoder: RepeatVector → LSTM(64) → Dropout → LSTM(128) → Dropout → Dense(6)
      Total parameters: 245,030
--------------------------------------------------------------------------------
TRAINING LSTM AUTOENCODER
--------------------------------------------------------------------------------


**Training Configuration**

In [10]:
print(f"   Configuration:")
print(f"      Epochs: {epochs}")
print(f"      Batch size: {batch_size}")
print(f"      Early stopping patience: 10")

        # Early stopping callback
early_stop = EarlyStopping(
  monitor='val_loss',
  patience=10,
  restore_best_weights=True
  )

        # Train
history = model.fit(
    X_train, X_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, X_val),
    callbacks=[early_stop],
    verbose=1
    )

print(f"\n   Training complete!")
print(f"     Final train loss: {history.history['loss'][-1]:.6f}")
print(f"     Final val loss: {history.history['val_loss'][-1]:.6f}")

   Configuration:
      Epochs: 50
      Batch size: 32
      Early stopping patience: 10
Epoch 1/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 97s 9ms/step - loss: 0.0028 - val_loss: 5.0688e-04
Epoch 2/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 76s 8ms/step - loss: 9.2864e-04 - val_loss: 3.5191e-04
Epoch 3/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 74s 8ms/step - loss: 7.9076e-04 - val_loss: 3.2854e-04
Epoch 4/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 82s 8ms/step - loss: 7.3230e-04 - val_loss: 3.3939e-04
Epoch 5/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 74s 8ms/step - loss: 7.0377e-04 - val_loss: 3.4741e-04
Epoch 6/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 75s 8ms/step - loss: 6.7938e-04 - val_loss: 3.2104e-04
Epoch 7/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 75s 8ms/step - loss: 6.5469e-04 - val_loss: 3.3813e-04
Epoch 8/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 74s 8ms/step - loss: 6.3922e-04 - val_loss: 3.7137e-04
Epoch 9/50
9443/9443 ━━━━━━━━━━━━━━━━━━━━ 75s 8ms/step - loss: 6.2855e-04 - val_loss: 2.9288e-04
Epoch 10/50
9443/9443 ━━━━━━━━━━━━━━━━━━━

**Computing Reconstruction Errors**

In [11]:
print("COMPUTING RECONSTRUCTION ERRORS")
print("-" * 80)

        # Reconstruct
X_pred = model.predict(X_sequences, verbose=0)

        # Calculate MSE for each sequence
reconstruction_errors = np.mean(np.square(X_sequences - X_pred), axis=(1, 2))

print(f"   Reconstruction errors computed: {len(reconstruction_errors)}")
print(f"   Min error: {reconstruction_errors.min():.6f}")
print(f"   Max error: {reconstruction_errors.max():.6f}")
print(f"   Mean error: {reconstruction_errors.mean():.6f}")
print(f"   Std error: {reconstruction_errors.std():.6f}")

COMPUTING RECONSTRUCTION ERRORS
--------------------------------------------------------------------------------
   Reconstruction errors computed: 377708
   Min error: 0.000002
   Max error: 0.044254
   Mean error: 0.000361
   Std error: 0.000603


**Anomaly Detection**

In [14]:
if threshold is None:
    threshold = np.percentile(reconstruction_errors, percentile)

print(f"Threshold (95th percentile): {threshold:.6f}")
print(f"Anomalies flagged: {(reconstruction_errors > threshold).sum()}")
print(f"Anomaly rate: {(reconstruction_errors > threshold).mean() * 100:.2f}%")

Threshold (95th percentile): 0.000977
Anomalies flagged: 18886
Anomaly rate: 5.00%


**Top 10 Highest Reconstruction Errors**

In [19]:
top_error_indices = np.argsort(reconstruction_errors)[-10:][::-1]
print("Top 10 highest reconstruction errors:")
for i, idx in enumerate(top_error_indices):
    print(f"  {i+1}. Index {idx}: error = {reconstruction_errors[idx]:.6f}")

Top 10 highest reconstruction errors:
  1. Index 8: error = 120495.000000
  2. Index 9: error = 120494.000000
  3. Index 6: error = 81794.000000
  4. Index 1: error = 81793.000000
  5. Index 2: error = 81792.000000
  6. Index 5: error = 81791.000000
  7. Index 0: error = 81788.000000
  8. Index 7: error = 81763.000000
  9. Index 3: error = 81762.000000
  10. Index 4: error = 81761.000000
